# 08 · Catálogo de paneles

> **En BigQuery.** Lee las cuatro tablas de deforestación y la tabla `REGISTRO_CARGAS` de `DATASET_DEFORESTACION`, y el catálogo de municipios del Observatorio. No escribe nada. **Qué tienes que cambiar:** nada; la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`). **Cuánto tarda:** unos cinco minutos.

Descripción breve de las cuatro tablas que produce el pipeline y de los
datos que contiene cada una.

Para el **qué y por qué** de cada fuente, ver
[`METODOLOGIA.md`](../guias/referencia/METODOLOGIA.md) (sección 1.2, *Para qué sirve cada
fuente*). Para el **cómo** se generan, ver
[`GUIA_CODIGO.md`](../guias/referencia/GUIA_CODIGO.md).

Las cifras de este cuaderno se leen de las tablas de BigQuery, así que se
actualizan solas al volver a ejecutarlo.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# --- Tablas del Observatorio (este notebook solo las lee) -------------
# Nombre completo: proyecto.dataset.tabla. Son las que se usaron en el
# trabajo de grado.
# Catálogo de municipios
TABLA_MUNICIPIOS = "prueba-ofr.Municipios.CODIGO"

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet folium==0.14.0 scipy matplotlib pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "08_eda_catalogo_paneles"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

In [ ]:
import json
import warnings

import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)


def cargar(nombre):
    """Lee un panel de DATASET_DEFORESTACION (los arman los notebooks 03 a 05)."""
    return leer_tabla(nombre)


gfw    = cargar("panel_deforestacion_colombia")
hansen = cargar("panel_hansen")
ideam  = cargar("panel_ideam")
dtd    = cargar("panel_dtd")

print("Paneles cargados.")

## Resumen

Los cuatro paneles cubren la **misma grilla de 5 km** y comparten el mismo
`bosque_base_ha`, el mismo cruce municipal y el mismo indexado de píxel a
celda. Esa base común es lo que permite compararlos entre sí.

Miden conceptos distintos, así que **sus cifras nunca se suman**.

In [ ]:
resumen = pd.DataFrame([
    {"panel": "panel_deforestacion_colombia", "fuente": "GFW",
     "mide": "Disturbio de la vegetación", "unidad": "hectáreas",
     "grano": "celda x mes"},
    {"panel": "panel_hansen", "fuente": "Hansen GFC",
     "mide": "Pérdida de cobertura arbórea", "unidad": "hectáreas",
     "grano": "celda x año"},
    {"panel": "panel_ideam", "fuente": "IDEAM / SMByC",
     "mide": "Deforestación de bosque natural", "unidad": "hectáreas",
     "grano": "celda x periodo"},
    {"panel": "panel_dtd", "fuente": "IDEAM / SMByC",
     "mide": "Detecciones de alerta temprana", "unidad": "conteo de puntos",
     "grano": "celda x trimestre"},
]).set_index("panel")

tablas = {"panel_deforestacion_colombia": gfw, "panel_hansen": hansen,
          "panel_ideam": ideam, "panel_dtd": dtd}
resumen["filas"] = [f"{len(tablas[p]):,}" for p in resumen.index]
resumen["columnas"] = [tablas[p].shape[1] for p in resumen.index]
resumen["celdas"] = [f"{tablas[p].cell_id.nunique():,}" for p in resumen.index]
resumen["municipios"] = [tablas[p].cod_dane.nunique() for p in resumen.index]
resumen

In [ ]:
# Verificacion de que la base comun es realmente comun.
celdas = {n: set(t.cell_id) for n, t in tablas.items()}
base = next(iter(celdas.values()))
bosque = {n: t.groupby("cell_id").bosque_base_ha.first() for n, t in tablas.items()}
ref = next(iter(bosque.values()))

print("mismas celdas en los cuatro :", all(c == base for c in celdas.values()))
print("diferencia max en bosque_base_ha:",
      max((v - ref).abs().max() for v in bosque.values()))
print(f"bosque base total           : {ref.sum():,.0f} ha")

---

## Recolección de los datos iniciales

Antes de describir cada panel (secciones 1 a 4) vale la pena documentar **cómo
llegaron los datos crudos**, porque las cuatro fuentes se recolectan con
métodos muy distintos entre sí -- API de pago por consulta, descarga masiva
pública, servidor institucional sin autenticación -- y esa diferencia explica
varias de las decisiones de calidad que aparecen más adelante (sección 8).

Los parámetros de cada fuente están en la celda 1 de su notebook de carga (02 a
05), y cada carga deja una fila en la tabla `REGISTRO_CARGAS` con los
parámetros que usó. Aquí se leen esas filas y se contrastan contra lo que hay
en cada tabla, como evidencia de que lo documentado y lo efectivamente
descargado coinciden.

**GFW (`PANEL_DEFORESTACION_COLOMBIA`) -- API de pago por consulta.**
Alertas del producto integrado `gfw_integrated_alerts` (GLAD-L + GLAD-S2 +
RADD), consultadas por lotes de ~400 celdas contra el endpoint asíncrono
`/query/batch` de la API SQL de GFW (`data-api.globalforestwatch.org`), con
una API key propia (celda 1 del notebook 03). Se filtra a confianza
`high`/`highest` (se excluye `nominal`, la menos confiable, por ser la menos
confiable de las tres). Cada lote se cachea en disco apenas se descarga
(en la carpeta de trabajo de la máquina), lo que hace la descarga reanudable
tras una interrupción.

**Hansen Global Forest Change -- descarga pública masiva, sin autenticación.**
Gránulos de 10°×10° (`treecover2000`, `lossyear`) alojados por Google en
Cloud Storage, descargados por bloques en paralelo con reintentos. Cumple dos
funciones: define el `bosque_base_ha` **común a las cuatro tablas** (notebook
02, con umbral de dosel de 30 % -- el de referencia más
usado en la literatura para operacionalizar "bosque" -- y corte en el año
2019) y aporta su propia medida de pérdida de cobertura arbórea.

**IDEAM / SMByC -- servidor institucional público.**
Dos productos del mismo servidor (`bart.ideam.gov.co`, solo HTTPS -- el
puerto 80 está cerrado y una URL `http://` no da error, simplemente se queda
esperando hasta agotar el tiempo de espera): las capas oficiales de cambio de
bosque natural (una por transición de dos años) y los boletines trimestrales
de detecciones tempranas (DTD, puntos sin huecos desde el tercer trimestre de
2016). Es la única fuente genuinamente **oficial** de las cuatro -- las otras
tres son de monitoreo/alerta, no de medición oficial, distinción que ya
aparece en las descripciones de las secciones 1 a 4.

**Verificación: los parámetros con que se cargó cada tabla
(`REGISTRO_CARGAS`) frente a lo que hay en ella.**


In [ ]:
# Parámetros con que se cargó cada tabla: la última fila de cada una en registro_cargas.
registro = leer_tabla("registro_cargas")
registro["tabla"] = registro["tabla"].str.lower()          # los nombres de tabla, sin importar mayúsculas
ultima = registro.sort_values("cargado_en").groupby("tabla").last()
param = {t: json.loads(ultima.loc[t, "parametros"]) for t in ultima.index}

print("Parámetros con que se cargó cada tabla (tabla REGISTRO_CARGAS):")
for t in ["panel_deforestacion_colombia", "panel_hansen", "panel_ideam", "panel_dtd"]:
    print(f"  {t}: {ultima.loc[t, 'fuente']}, cargada el {ultima.loc[t, 'cargado_en']:%Y-%m-%d %H:%M} UTC")
    print(f"    {param[t]}")
print()

print("Lo que realmente hay en las tablas:")
print(f"  GFW     periodo: {gfw.periodo.min():%Y-%m} a {gfw.periodo.max():%Y-%m}")
print(f"  Hansen  anios con perdida_ha > 0: {sorted(hansen.loc[hansen.perdida_ha > 0, 'anio'].unique().tolist())}")
print(f"  IDEAM   periodos presentes: {sorted(ideam.periodo.unique().tolist())}")
print(f"  DTD     trimestres: {dtd.trimestre_txt.min()} a {dtd.trimestre_txt.max()} "
      f"({dtd.trimestre_txt.nunique()} trimestres)")
print()

p_gfw = param["panel_deforestacion_colombia"]
coincide_ideam = set(param["panel_ideam"]["periodos"]) == set(ideam.periodo.unique())
coincide_dtd_anios = set(dtd.anio.unique()) <= set(param["panel_dtd"]["anios"])
coincide_gfw_inicio = gfw.periodo.min() == pd.Timestamp(p_gfw["desde"])
coincide_gfw_fin = gfw.periodo.max() < pd.Timestamp(p_gfw["hasta_sin_incluir"])

print("coincide IDEAM (declarado vs. panel)         :", coincide_ideam)
print("coincide DTD (anios del panel dentro de lo pedido):", coincide_dtd_anios)
print("coincide inicio de ventana GFW               :", coincide_gfw_inicio)
print("coincide fin (exclusivo) de ventana GFW      :", coincide_gfw_fin)

**Hallazgos**

- Las cuatro fuentes se recolectan con métodos distintos (API de pago,
  descarga pública masiva, servidor institucional) pero con un patrón común:
  todo parámetro que afecta el resultado vive en `Config` -- nunca
  *hardcodeado* dentro de un script -- y la descarga es reanudable (cache por
  lote, por gránulo o por archivo), lo que hace la recolección reproducible.
  Esto es evidencia concreta del "rigor metodológico" que la síntesis de este
  mismo cuaderno ya cita como uno de los principios del Observatorio (sección
  10.1, más abajo).
- La ventana y los periodos declarados en `Config` coinciden exactamente con
  lo que contiene el panel ya construido (las cuatro comprobaciones de la
  celda anterior dan `True`), lo que confirma que no hay desalineación entre
  lo documentado en el código y lo efectivamente descargado -- es decir, que
  la "recolección de los datos iniciales" no vive solo en la intención del
  script, sino que se puede verificar contra el resultado real.
- La fecha de inicio de la ventana de GFW (2020-01-01) no es arbitraria:
  `config_local.py` fija los periodos de IDEAM a partir de la fecha de corte
  de la EUDR (31-dic-2020), así que la ventana de recolección ya queda
  alineada con el estándar regulatorio que el documento de entendimiento del
  negocio usa como referencia para "deforestación libre" (ver el objetivo de
  minería de datos 1, sección 10.1).


---

## 1. `PANEL_DEFORESTACION_COLOMBIA` — alertas de GFW

**Qué contiene.** Hectáreas con alerta de disturbio de la vegetación por
celda y mes, a partir del producto integrado `gfw_integrated_alerts`
(GLAD-L, GLAD-S2 y RADD), filtrado a confianza alta y muy alta.

**Para qué sirve.** Responde **cuándo** está ocurriendo el fenómeno. Es el
de mayor resolución temporal y el insumo previsto para el modelo.

**Panel balanceado**: toda celda tiene una fila en todo periodo, de modo
que `filas = celdas × periodos` exactamente.

In [ ]:
print(f"periodos      : {gfw.periodo.min():%Y-%m} a {gfw.periodo.max():%Y-%m} "
      f"({gfw.periodo.nunique()} meses)")
print(f"balanceado    : {len(gfw) == gfw.cell_id.nunique() * gfw.periodo.nunique()}")
print(f"area con alerta: {gfw.area_def_ha.sum():,.0f} ha")
print(f"celda-mes con evento: {100 * gfw.evento.mean():.2f} %")
print()
print("Columnas:")
for i, c in enumerate(gfw.columns, 1):
    print(f"  {i:2d}. {c:<22} {gfw[c].dtype}")

In [ ]:
gfw[gfw.evento == 1].head(3)

---

## 2. `PANEL_HANSEN` — pérdida de cobertura arbórea

**Qué contiene.** Hectáreas que perdieron cobertura arbórea por celda y
año, desde la capa `lossyear` de Hansen Global Forest Change.

**Para qué sirve.** Responde **cómo se compara Colombia con otros
países**: Hansen aplica el mismo algoritmo en todo el planeta. El mismo
producto aporta además la línea base `bosque_base_ha` que usan los cuatro
paneles.

**Ojo con el concepto**: la pérdida de cobertura incluye cosecha de
plantación forestal, incendio y daño natural, así que su cifra es mayor
que la deforestación oficial.

In [ ]:
print("Pérdida de cobertura por año (ha):")
print(hansen.groupby("anio").perdida_ha.sum().apply(lambda x: f"{x:,.0f}").to_string())
print()
print("Columnas:")
for i, c in enumerate(hansen.columns, 1):
    print(f"  {i:2d}. {c:<22} {hansen[c].dtype}")

---

## 3. `PANEL_IDEAM` — deforestación oficial

**Qué contiene.** Las cinco clases de las capas de cambio del SMByC por
celda y periodo, en hectáreas. La clase de interés es `def_ha`
(deforestación); las demás sirven de contexto y control de calidad.

**Para qué sirve.** Responde **cuánto**, con la cifra oficial de Colombia
y la definición nacional de bosque.

**Dos advertencias sobre esta tabla:**

- `periodo` designa una **transición** entre dos composiciones anuales de
  imágenes, y su cifra corresponde al **año final**: `2022-2023` produce
  el dato oficial de 2023.
- El denominador correcto para sus tasas es **`bosque_ideam_ha`** (bosque
  natural), no `bosque_base_ha` (Hansen, que incluye plantaciones).

In [ ]:
col = ["def_ha", "bosque_estable_ha", "regeneracion_ha", "sin_info_ha"]
por_periodo = ideam.groupby("periodo")[col].sum()

# Cifra oficial publicada por el IDEAM, para contrastar.
OFICIAL = {"2020-2021": 174_103, "2021-2022": 123_517,
           "2022-2023": 79_256, "2023-2024": 113_608,
           "2024-2025": 119_483}
por_periodo["oficial_ha"] = pd.Series(OFICIAL)
por_periodo["recuperado_%"] = (100 * por_periodo.def_ha / por_periodo.oficial_ha).round(2)

# Las hectareas se redondean a entero; el porcentaje conserva decimales
# porque la cercania al 100 % es justamente lo que valida la agregacion.
por_periodo.astype({c: "int64" for c in col + ["oficial_ha"]}, errors="ignore")

In [ ]:
print("Los dos denominadores, a escala nacional:")
b = ideam[ideam.periodo == ideam.periodo.min()]
print(f"  bosque_base_ha  (Hansen, dosel >= 30 %) : {b.bosque_base_ha.sum():>14,.0f} ha")
print(f"  bosque_ideam_ha (bosque natural, IDEAM) : {b.bosque_ideam_ha.sum():>14,.0f} ha")
print()
print("Columnas:")
for i, c in enumerate(ideam.columns, 1):
    print(f"  {i:2d}. {c:<22} {ideam[c].dtype}")

---

## 4. `PANEL_DTD` — alertas tempranas oficiales

**Qué contiene.** Número de puntos de alerta que el IDEAM publicó por
celda y trimestre, más cuántos de ellos caen en un área protegida del
SINAP.

**Para qué sirve.** Responde **dónde**, con respaldo de la autoridad
ambiental. Es el único producto oficial de cadencia sub-anual disponible
para Colombia.

**La unidad es el conteo, no la hectárea.** Los datos son puntos sin
superficie asociada: marcan un sitio donde se detectó un cambio
compatible con deforestación, sin cuantificar su extensión.

**Ningún agregado de esta tabla sirve como serie temporal.** El criterio
de detección cambió a lo largo de la serie, así que los conteos por año
se mueven en sentido contrario a la deforestación real. La tabla sirve
**dentro** de cada trimestre, para comparar unos lugares con otros.

In [ ]:
print(f"trimestres: {dtd.trimestre_txt.min()} a {dtd.trimestre_txt.max()} "
      f"({dtd.trimestre_txt.nunique()})")
print(f"balanceado: {len(dtd) == dtd.cell_id.nunique() * dtd.trimestre_txt.nunique()}")
print()
print("Detecciones por año:")
print(dtd.groupby("anio").n_alertas.sum().apply(lambda x: f"{x:,}").to_string())
print()

# sinap_disponible marca los trimestres en que el IDEAM informa el campo:
# un vacio de informacion y una ausencia de alertas son cosas distintas.
con = dtd[dtd.sinap_disponible]
print(f"En areas protegidas del SINAP: {int(con.n_alertas_sinap.sum()):,} de "
      f"{con.n_alertas.sum():,} ({100 * con.n_alertas_sinap.sum() / con.n_alertas.sum():.1f} %), "
      f"en {con.trimestre_txt.nunique()} de {dtd.trimestre_txt.nunique()} trimestres")
print()
print("Columnas:")
for i, c in enumerate(dtd.columns, 1):
    print(f"  {i:2d}. {c:<22} {dtd[c].dtype}")

---

## Las tres fuentes en hectáreas, lado a lado

Las tres primeras tablas miden superficie y son comparables en magnitud.
Las razones frente a la cifra oficial **varían año a año**, así que
convertir una fuente en otra exige un factor específico para cada año.

In [ ]:
anual = pd.DataFrame({
    "GFW_ha": gfw.groupby("anio").area_def_ha.sum(),
    "Hansen_ha": hansen.groupby("anio").perdida_ha.sum(),
    "IDEAM_ha": (ideam.assign(anio=ideam.periodo.str[-4:].astype(int))
                      .groupby("anio").def_ha.sum()),
})
anual["oficial_ha"] = pd.Series(
    {2021: 174_103, 2022: 123_517, 2023: 79_256,
     2024: 113_608, 2025: 119_483})
anual["Hansen/IDEAM"] = (anual.Hansen_ha / anual.IDEAM_ha).round(2)
anual["GFW/IDEAM"] = (anual.GFW_ha / anual.IDEAM_ha).round(2)

for c in ["GFW_ha", "Hansen_ha", "IDEAM_ha", "oficial_ha"]:
    anual[c] = anual[c].round(0)
anual

---

## Cómo se ve cada fuente en el mapa

Un mapa con una capa por fuente, para comparar su patrón espacial sobre
todo el país.

El mapa base es el gris claro de Esri, que no pide llave, atenuado con un filtro CSS
para que los datos resalten.

In [ ]:
import folium
from folium.plugins import Fullscreen, HeatMap

CENTRO_COLOMBIA = (4.2, -73.5)
# Mapa base: el gris claro de Esri, que no pide llave. OpenStreetMap y
# CARTO no sirven en BigQuery: el primero bloquea los mapas que no dicen
# desde qué página se piden y el segundo ya exige API key.
TILES_DEFECTO = ("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
                 "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")
ATRIBUCION_TILES = "Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"


def atenuar_mapa_base(mapa):
    """Mapa base gris y tenue, para que resalten los datos (filtro CSS del navegador)."""
    mapa.get_root().header.add_child(folium.Element(
        "<style>.leaflet-tile-pane{filter:grayscale(100%) brightness(107%) contrast(88%);}</style>"))

# Cada panel se colapsa a UNA fila por celda, sumando toda su ventana.
# Asi las cuatro fuentes quedan sobre la misma geometria y se pueden
# superponer como capas del mismo mapa.
por_celda = {
    "GFW — disturbio (ha)":        gfw.groupby("cell_id").area_def_ha.sum(),
    "Hansen — pérdida (ha)":       hansen.groupby("cell_id").perdida_ha.sum(),
    "IDEAM — deforestación (ha)":  ideam.groupby("cell_id").def_ha.sum(),
    "DTD — detecciones (conteo)":  dtd.groupby("cell_id").n_alertas.sum(),
}

# Coordenadas de cada celda, iguales en los cuatro paneles.
coords = gfw.groupby("cell_id")[["lat", "lon"]].first()

for nombre, serie in por_celda.items():
    activas = (serie > 0).sum()
    print(f"{nombre:<30} {activas:>6,} celdas con dato "
          f"({100 * activas / len(serie):>4.1f} %)")

### Una capa por fuente

Cada capa es un mapa de calor con el peso **normalizado dentro de su
propia fuente**. Eso es deliberado: las unidades no son comparables
(hectáreas frente a conteos), pero el **patrón espacial** sí, y es lo que
interesa contrastar aquí. Enciende y apaga capas para ver si los focos
coinciden.

Para no producir un archivo de decenas de MB, cada capa se limita a sus
celdas de mayor actividad. El patrón nacional se conserva; para el detalle
de las alertas de GFW está el mapa del notebook 03.


In [ ]:
TOPE = 4000   # celdas por capa: suficiente para el patron, liviano de abrir

# Un color por fuente, para distinguirlas al alternar capas.
GRADIENTES = {
    "GFW — disturbio (ha)":       {0.3: "#FBE3CE", 0.6: "#E2905A", 1.0: "#7A3319"},
    "Hansen — pérdida (ha)":      {0.3: "#FDE68A", 0.6: "#D97706", 1.0: "#78350F"},
    "IDEAM — deforestación (ha)": {0.3: "#FECDD3", 0.6: "#E11D48", 1.0: "#701A29"},
    "DTD — detecciones (conteo)": {0.3: "#C7D2FE", 0.6: "#4F46E5", 1.0: "#26216B"},
}

mapa = folium.Map(location=CENTRO_COLOMBIA, zoom_start=5,
                  tiles=TILES_DEFECTO, attr=ATRIBUCION_TILES, control_scale=True,
                  prefer_canvas=True)
atenuar_mapa_base(mapa)

for i, (nombre, serie) in enumerate(por_celda.items()):
    top = serie[serie > 0].nlargest(TOPE)
    xy = coords.loc[top.index]
    # Peso relativo al maximo de ESTA fuente, no al de todas.
    peso = (top / top.max()).round(3)
    HeatMap(
        list(zip(xy.lat.round(4), xy.lon.round(4), peso)),
        name=nombre, radius=10, blur=15, max_zoom=9, min_opacity=.3,
        gradient=GRADIENTES[nombre],
        show=(i == 0),          # solo la primera encendida al abrir
    ).add_to(mapa)

# Bosque base como contexto: donde habia bosque para empezar.
bb = ideam.groupby("cell_id").bosque_base_ha.first()
top_bb = bb.nlargest(TOPE)
xy = coords.loc[top_bb.index]
HeatMap(list(zip(xy.lat.round(4), xy.lon.round(4),
                 (top_bb / top_bb.max()).round(3))),
        name="Bosque base (contexto)", radius=9, blur=14, max_zoom=9,
        min_opacity=.15, show=False,
        gradient={0.2: "#2A3327", 0.6: "#4C5A4E", 1.0: "#1E6B45"}).add_to(mapa)

folium.LayerControl(collapsed=False).add_to(mapa)
Fullscreen(position="topleft").add_to(mapa)


mapa

Los focos de las tres fuentes en hectáreas se concentran en el mismo
arco amazónico —Caquetá, Meta, Guaviare, Putumayo—, y el de las
detecciones oficiales coincide con ellos. Donde más se separan es en la
zona andina y el Caribe, con sensibilidades distintas según la fuente.

El mapa va limitado en número de celdas para que el cuaderno siga siendo
liviano. Para la vista completa del panel de alertas, con agrupación de
marcadores, escala de color y detalle por celda, está la sección 10 del
notebook 03.

---

## Cómo cruzarlos

Las cuatro tablas comparten `cell_id` y `cod_dane`, de modo que el cruce
es directo. Al agregar por municipio conviene usar `cod_dane` como llave,
nunca el nombre: las tildes y mayúsculas no son consistentes entre
fuentes.

In [ ]:
COD = "18150"   # Cartagena del Chaira, Caqueta

print(f"Municipio {COD} — {ideam[ideam.cod_dane == COD].municipio.iloc[0]}")
print(f"  GFW    (disturbio, ha)      : {gfw[gfw.cod_dane == COD].area_def_ha.sum():>10,.0f}")
print(f"  Hansen (pérdida, ha)        : {hansen[hansen.cod_dane == COD].perdida_ha.sum():>10,.0f}")
print(f"  IDEAM  (deforestación, ha)  : {ideam[ideam.cod_dane == COD].def_ha.sum():>10,.0f}")
print(f"  DTD    (detecciones)        : {dtd[dtd.cod_dane == COD].n_alertas.sum():>10,}")

In [ ]:
# Ranking municipal segun la fuente oficial, que es el uso previsto
# para el tamizaje de riesgo.
(ideam.groupby(["cod_dane", "municipio", "departamento"])
      .def_ha.sum().sort_values(ascending=False)
      .head(10).round(0).reset_index())

---

## 5. Distribución de cada fuente (EDA — parte 1)

Hasta acá el catálogo mostraba **totales y rankings** (cuánto por año, cuáles
municipios encabezan la lista). Lo que falta, y que pidió explícitamente el
tutor ("analizar datos y declarar si se necesita limpieza"), es mirar la
**distribución** de la medida principal de cada fuente: qué tan dispersa es,
qué tan sesgada, y si hay valores atípicos que ameriten revisión antes de
usarla para cruzar con las variables financieras.

Se usa la medida principal de cada tabla, al nivel más fino (celda x periodo):

| Fuente | Medida | Columna |
|---|---|---|
| GFW | disturbio | `area_def_ha` |
| Hansen | pérdida de cobertura | `perdida_ha` |
| IDEAM | deforestación oficial | `def_ha` |
| DTD | detecciones tempranas | `n_alertas` |


In [ ]:
MEDIDA = {
    "GFW":    (gfw,    "area_def_ha"),
    "Hansen": (hansen, "perdida_ha"),
    "IDEAM":  (ideam,  "def_ha"),
    "DTD":    (dtd,    "n_alertas"),
}

# Estadisticos basicos a nivel celda x periodo, ANTES de graficar: dan una
# primera senal de sesgo (p99 muy por encima de la mediana) y de cuantos
# registros son ceros (celda-periodo sin evento, lo esperado en un panel de
# alertas -- no es un dato faltante, es la ausencia real del evento).
filas = []
for fuente, (tabla, col) in MEDIDA.items():
    s = tabla[col]
    filas.append({
        "fuente": fuente,
        "columna": col,
        "% ceros": round(100 * (s == 0).mean(), 2),
        "media": round(s.mean(), 4),
        "p50": round(s.quantile(0.50), 4),
        "p95": round(s.quantile(0.95), 4),
        "p99": round(s.quantile(0.99), 4),
        "max": round(s.max(), 2),
    })
pd.DataFrame(filas).set_index("fuente")

In [ ]:
# Paleta y utilidades de estilo para los graficos de esta seccion: un color
# fijo por fuente (nunca reciclado), una paleta secuencial de azules para
# magnitudes de una sola fuente, y dos ayudas de formato (limpiar_ejes,
# nombre_propio) que se reutilizan en el resto de graficos del EDA.
import matplotlib as mpl

BLUE    = "#2a78d6"   # GFW
ORANGE  = "#eb6834"   # Hansen
AQUA    = "#1baf7a"   # IDEAM
YELLOW  = "#eda100"   # DTD
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#0d366b"]
GOOD = "#0ca30c"
CRITICAL = "#d03b3b"
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
FUENTE_COLOR = {"GFW": BLUE, "Hansen": ORANGE, "IDEAM": AQUA, "DTD": YELLOW}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK_PRIMARY, "text.color": INK_PRIMARY,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "font.size": 10.5,
})

def limpiar_ejes(ax, grid_eje="y"):
    """Deja solo la grilla principal y quita marcas de eje sobrantes."""
    ax.grid(axis=grid_eje, which="major")
    ax.grid(False, axis=("x" if grid_eje == "y" else "y"))
    ax.tick_params(length=0)
    ax.spines["bottom"].set_color(AXIS)

_CONECTORES = {"de", "del", "la", "las", "los", "y"}
def nombre_propio(s):
    """'Cartagena Del Chairá' -> 'Cartagena del Chairá' (respeta conectores)."""
    palabras = str(s).split()
    out = []
    for i, p in enumerate(palabras):
        if i > 0 and p.lower() in _CONECTORES:
            out.append(p.lower())
        else:
            out.append(p[:1].upper() + p[1:].lower() if p.isupper() or p.islower() else p)
    return " ".join(out)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Histograma de los valores DISTINTOS DE CERO: con tantos ceros (celdas sin
# evento ese periodo) un histograma normal se ve como una sola barra. Lo que
# interesa aqui es la forma de la distribucion CUANDO SI hay evento -- y para
# eso ayuda el eje Y en escala log, porque se espera cola larga (pocas celdas
# con mucha area/perdida/alertas, muchas con poca). La linea roja marca el
# percentil 95 de cada fuente (ya calculado en la tabla de arriba).
#
# ESCALA COMUN (pedido explicito: "las graficas deben tener la misma
# escala"): GFW, Hansen e IDEAM miden la MISMA unidad (hectareas), asi que
# comparten el mismo rango de eje X y los mismos bins -- el ancho y la
# posicion de cada barra quedan directamente comparables entre las tres.
# DTD mide algo distinto (conteo de puntos de alerta, no hectareas):
# ponerlo en ese mismo eje numerico no seria "la misma escala", seria
# comparar unidades que no son comparables (su histograma quedaria
# aplastado contra el cero, sin poderse leer). Por eso DTD conserva su
# propio rango -- decision que se deja explicita aqui en vez de dejarla
# en el auto-escalado de matplotlib, igual que las demas decisiones de
# este cuaderno (ver 8.4).
HECTAREAS = ["GFW", "Hansen", "IDEAM"]
valores_por_fuente = {f: tabla.loc[tabla[col] > 0, col] for f, (tabla, col) in MEDIDA.items()}
max_ha = max(valores_por_fuente[f].max() for f in HECTAREAS)
bins_ha = np.linspace(0, max_ha, 41)

fig, ejes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (fuente, (tabla, col)) in zip(ejes.flat, MEDIDA.items()):
    valores = valores_por_fuente[fuente]
    bins = bins_ha if fuente in HECTAREAS else 40
    ax.hist(valores, bins=bins, color=FUENTE_COLOR[fuente], edgecolor="white", linewidth=0.4)
    ax.set_yscale("log")
    if fuente in HECTAREAS:
        ax.set_xlim(0, max_ha)
    p95 = valores.quantile(0.95)
    ax.axvline(p95, color=CRITICAL, linestyle="--", linewidth=1.2)
    ax.text(p95, ax.get_ylim()[1], f" p95={p95:,.1f}", color=CRITICAL, fontsize=8.5, ha="left", va="top")
    ax.set_title(f"{fuente} — n={len(valores):,} celdas > 0", fontsize=11, color=INK_PRIMARY, loc="left")
    ax.set_xlabel(col, fontsize=9.5)
    ax.set_ylabel("celdas (escala log)", fontsize=9.5)
    limpiar_ejes(ax)
fig.suptitle("Distribución de la medida principal por fuente, a nivel celda x periodo",
             x=0.03, y=0.99, ha="left", fontsize=14, fontweight="bold", color=INK_PRIMARY)
fig.text(0.03, 0.945, "Solo valores > 0 (celda-periodo con evento); línea roja = percentil 95.",
          fontsize=10, color=INK_SECONDARY)
fig.text(0.03, 0.918, "Escala común: GFW/Hansen/IDEAM comparten eje X (hectáreas, mismos bins); "
          "DTD queda aparte (otra unidad: conteo).",
          fontsize=9, color=INK_MUTED, style="italic")
fig.tight_layout(rect=[0, 0, 1, 0.88])

### Agregado por municipio

La misma medida, pero sumada por `cod_dane` a lo largo de toda la ventana
temporal: muestra qué tan concentrada está la señal en pocos municipios
frente a estar repartida de forma pareja -- relevante para el KPI que pidió
el tutor de cruzar desforestación con otras variables a nivel municipal.

In [ ]:
# ESCALA COMUN: mismo criterio que en la figura anterior -- GFW, Hansen e
# IDEAM (hectareas) comparten eje X y bins; DTD (conteo) conserva el suyo.
por_municipio_por_fuente = {f: tabla.groupby("cod_dane")[col].sum() for f, (tabla, col) in MEDIDA.items()}
max_ha_municipio = max(por_municipio_por_fuente[f].max() for f in HECTAREAS)
bins_ha_municipio = np.linspace(0, max_ha_municipio, 41)

fig, ejes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (fuente, (tabla, col)) in zip(ejes.flat, MEDIDA.items()):
    por_municipio = por_municipio_por_fuente[fuente]
    bins = bins_ha_municipio if fuente in HECTAREAS else 40
    ax.hist(por_municipio, bins=bins, color=FUENTE_COLOR[fuente], edgecolor="white", linewidth=0.4)
    ax.set_yscale("log")
    if fuente in HECTAREAS:
        ax.set_xlim(0, max_ha_municipio)

    top_cod = por_municipio.idxmax()
    top_val = por_municipio.max()
    top_nombre = nombre_propio(tabla.loc[tabla.cod_dane == top_cod, "municipio"].iloc[0])
    ax.axvline(top_val, color=CRITICAL, linestyle="--", linewidth=1.2)
    ax.annotate(f"{top_nombre}\n{top_val:,.0f}", xy=(top_val, ax.get_ylim()[1] * 0.5),
                xytext=(-10, 0), textcoords="offset points", ha="right", va="center",
                fontsize=8.7, color=CRITICAL, fontweight="bold")

    ax.set_title(f"{fuente} — {por_municipio.shape[0]:,} municipios", fontsize=11, color=INK_PRIMARY, loc="left")
    ax.set_xlabel(f"{col} (total del municipio)", fontsize=9.5)
    ax.set_ylabel("municipios (escala log)", fontsize=9.5)
    limpiar_ejes(ax)
fig.suptitle("Concentración municipal de cada fuente", x=0.03, y=0.99, ha="left",
             fontsize=14, fontweight="bold", color=INK_PRIMARY)
fig.text(0.03, 0.945, "Total acumulado 2020-2026 por municipio; línea roja = municipio con mayor total.",
          fontsize=10, color=INK_SECONDARY)
fig.text(0.03, 0.918, "Escala común: GFW/Hansen/IDEAM comparten eje X (hectáreas, mismos bins); "
          "DTD queda aparte (otra unidad: conteo).",
          fontsize=9, color=INK_MUTED, style="italic")
fig.tight_layout(rect=[0, 0, 1, 0.88])

### Dispersión por departamento (fuente oficial IDEAM)

Boxplot de `def_ha` sumado por municipio, agrupado por departamento, para
los departamentos con mayor deforestación acumulada. Un boxplot muestra de
un vistazo la mediana, el rango intercuartílico y los municipios atípicos
dentro de cada departamento -- útil para decidir si algún outlier amerita
revisión manual antes del cruce.

In [ ]:
TOP_DEPTOS = 8

por_muni_dpto = (ideam.groupby(["cod_dane", "departamento"]).def_ha.sum()
                       .reset_index())
orden = (por_muni_dpto.groupby("departamento").def_ha.sum()
                       .sort_values(ascending=False).head(TOP_DEPTOS).index)

datos_boxplot = [por_muni_dpto.loc[por_muni_dpto.departamento == d, "def_ha"] for d in orden]
etiquetas = [nombre_propio(d) for d in orden]
medianas = [d.median() for d in datos_boxplot]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.boxplot(datos_boxplot, tick_labels=etiquetas, showfliers=True, patch_artist=True,
           widths=0.55,
           boxprops=dict(facecolor=SEQ[2], edgecolor=SEQ[5], linewidth=1.1),
           medianprops=dict(color=SEQ[6], linewidth=1.8),
           whiskerprops=dict(color=SEQ[5], linewidth=1.1),
           capprops=dict(color=SEQ[5], linewidth=1.1),
           flierprops=dict(marker="o", markerfacecolor=CRITICAL, markeredgecolor="none",
                            markersize=4, alpha=0.55))

i_max = int(pd.Series(medianas).idxmax())
ax.annotate(f"mediana {medianas[i_max]:,.0f} ha", xy=(i_max + 1, medianas[i_max]),
            xytext=(12, 14), textcoords="offset points", fontsize=9.5,
            color=SEQ[6], fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=SEQ[6], linewidth=1))

ax.set_ylabel("def_ha acumulado por municipio (2020-2026)", fontsize=10)
limpiar_ejes(ax)
plt.xticks(rotation=25, ha="right")
fig.suptitle(f"Dispersión municipal de deforestación IDEAM — top {TOP_DEPTOS} departamentos",
             x=0.03, y=0.99, ha="left", fontsize=14, fontweight="bold", color=INK_PRIMARY)
fig.text(0.03, 0.925, "Cada punto rojo es un municipio atípico dentro de su departamento (posible caso a revisar).",
          fontsize=10, color=INK_SECONDARY)
fig.tight_layout(rect=[0, 0, 1, 0.88])

### Hallazgos

- **% de ceros por fuente:** GFW 71.8%, Hansen 51.7%, IDEAM 77.9%, DTD 95.1%.
  Es consistente con lo esperado para un panel de alertas (la mayoria de
  celda-periodo sin evento) -- DTD es la mas dispersa y probablemente conviene
  mirarla siempre agregada (por municipio o periodo), no a nivel celda.
- **Sesgo / outliers:** las cuatro distribuciones son fuertemente sesgadas a
  la derecha. A nivel departamental, Guaviare tiene una caja alta y compacta
  (deforestacion consistentemente alta), mientras que Caqueta y Meta tienen
  municipios outlier extremos (Cartagena del Chaira ~55,725 ha, San Vicente
  del Caguan ~44,512 ha acumuladas). Otros departamentos tienen outliers
  puntuales, no generalizados.
- **¿Se necesita limpieza?** No se encontro evidencia de errores de datos --
  el sesgo es una caracteristica real del fenomeno (la deforestacion se
  concentra en pocos municipios/celdas), no un problema de calidad. La
  implicacion es para el modelado (zero-inflation, transformacion log), no
  para limpieza de los datos.


---

## 6. Llave municipal: compatibilidad con las tablas del Observatorio

El profesor Jairo autorizo el 14/09/2026 acceso de solo lectura a tres tablas
del Observatorio (proyecto `prueba-ofr`): `FINAGRO_Desembolsos_EFECTIVA`,
`SFC414_DASH` y `Municipios.CODIGO`. Se describieron con
`describir_fuentes_profesor.py` (cuenta personal de Google via ADC, mientras
el profesor autoriza tambien a la cuenta de servicio) -- ver
`DESCRIPCION_FUENTES_PROFESOR.md` para columnas y muestra completas de cada
una.

| Tabla | Filas | Columnas | Columna de municipio |
|---|---|---|---|
| `FINAGRO_Desembolsos_EFECTIVA` | 6,158,859 | 37 | `cod_municipio` (STRING) |
| `SFC414_DASH` | 1,948,973 | 17 | `Codigo_Municipio` (STRING) |
| `Municipios.CODIGO` | 20,196 | 6 | `cod_dane` (STRING) |

Antes de cruzar esas tablas con nuestro panel habia que resolver uno de los
pendientes del tutor: confirmar si la llave municipal (`cod_dane`) es
compatible entre ambos lados. Las tres tablas del profesor guardan el codigo
de municipio como **STRING, con el cero a la izquierda conservado**
(`"05001"` para Medellin). En nuestro panel, `cargar()` (celda de arriba del
todo) ya declara `cod_dane` como texto al leer el CSV por la misma razon --
pero la tabla equivalente subida a BigQuery (`staging.stg_*`, ver
`DESCRIPCION_FUENTES.md`) quedo como **INTEGER**, que pierde ese cero. Es
decir: en pandas (este notebook) el cruce ya es seguro; un cruce hecho
directamente en BigQuery entre `staging.*` y las tablas del profesor
necesitaria primero convertir `cod_dane` a texto de 5 caracteres con
`LPAD(CAST(cod_dane AS STRING), 5, '0')`.

La celda de abajo lo verifica con datos reales: cuenta cuantos municipios de
nuestro panel tienen match, como texto, con el catalogo `Municipios.CODIGO`
del profesor.


In [ ]:
try:
    municipios_profe = leer_sql(f"SELECT DISTINCT cod_dane FROM `{TABLA_MUNICIPIOS}`")["cod_dane"]

    propios = set(ideam["cod_dane"].unique())
    ajenos = set(municipios_profe)

    print(f"Municipios propios (panel IDEAM):                {len(propios)}")
    print(f"Municipios en la tabla de municipios del Observatorio: {len(ajenos)}")
    print(f"En comun (cruzando como texto, cero a la izq.):  {len(propios & ajenos)}")
    print(f"Propios sin match en la tabla del Observatorio:  {len(propios - ajenos)}")
    if propios - ajenos:
        print("Ejemplos sin match:", sorted(propios - ajenos)[:10])
except Exception as e:
    print("No se pudo consultar la tabla de municipios del Observatorio. Detalle:", e)

### Hallazgos

- **Match casi total:** de los 1,114 municipios de nuestro panel, **1,113
  (99.9%)** cruzan sin problema con el catalogo `Municipios.CODIGO` del
  profesor al tratar `cod_dane` como texto de 5 caracteres. Esto confirma
  que el formato (INTEGER vs. STRING / cero a la izquierda) era el unico
  riesgo real, y que ya esta resuelto en pandas gracias a `cargar()`.
- **Unico caso sin match -- investigado y resuelto (11/10/2026):**
  `cod_dane = 27493` es Nuevo Belen de Bajira (Choco), el municipio mas
  nuevo de Colombia (creado administrativamente en 2022, con gobierno
  propio desde junio de 2024). No es un error de formato ni de nuestros
  datos: el catalogo `Municipios.CODIGO` del profesor, generado antes de
  esa fecha, simplemente no lo incluye -- se confirmo por codigo y por
  nombre, sobre la tabla completa (20,196 filas), sin ningun match.
  Investigacion completa, con las consultas y resultados reales, en
  `procesamiento y eda/investigacion_municipio_27493.ipynb (en el repositorio)` (cuaderno aparte, no
  forma parte de este EDA).

  El punto central: el municipio **si tiene datos reales** en las tablas
  del profesor -- 30 filas de produccion agropecuaria en EVA (todas de
  2024), 197 creditos de FINAGRO por $5.298 millones COP, y 202 registros
  de SFC414 (348 creditos) por $6.717 millones COP -- pero cero filas en
  ICA pecuario. **Decision:** tratar el vacio como un faltante estructural
  documentado, limitado a las variables que dependen del cruce con
  `Municipios.CODIGO` (departamento, region, ruralidad, PDET), que quedan
  en NaN para este municipio -- sin imputar ni rellenar con un valor
  departamental, y sin tocar los valores de credito/produccion reales que
  si existen. Se descarto rellenar con el agregado de Choco-departamento
  (la alternativa que sugirio originalmente el profesor Jairo) porque
  diluiria datos reales y precisos en un promedio departamental que ya
  sabemos que no hace falta.

  `06_carga_panel_municipal` ya implementa esto correctamente, sin
  necesitar ningun cambio: `codigos = municipios.index.union(fijas.index)`
  mantiene el municipio en el panel (via nuestro propio panel de
  deforestacion, no via `Municipios.CODIGO`), y el `.join(municipios)`
  final es un left join -- el credito y la produccion reales se preservan,
  solo las columnas que vienen de `Municipios.CODIGO` salen NaN para este
  caso.
- **¿Se necesita normalizar antes del cruce con FINAGRO/SFC414?** Si, pero
  solo al cruzar directamente en BigQuery (`staging.*` quedo como INTEGER
  ahi) -- en pandas, siempre y cuando se siga usando `cargar()` (que ya
  declara `cod_dane` como texto), el cruce con las tablas del profesor es
  seguro tal cual.


---

## 7. Correlación entre fuentes de deforestación

Punto 6 de la lista del tutor y foco de la semana 8 del cronograma: cuantificar
qué tanto coinciden las cuatro fuentes entre sí, en vez de solo mirarlas por
separado (secciones anteriores) o comparar cifras nacionales año a año sin
un número que resuma el acuerdo. Se mira desde dos ángulos, porque las
fuentes miden cosas distintas y a distinta granularidad temporal (celda 1,
arriba: GFW es mensual, Hansen es anual, IDEAM es bienal, DTD es trimestral):

1. **En el tiempo** (nivel nacional): ¿suben y bajan juntas, año a año? Se
   agregan las cuatro a año calendario (IDEAM se asigna al año en que
   termina su periodo bienal, igual que en "Cifras nacionales por año").
2. **En el espacio** (nivel municipio): ¿los municipios que una fuente marca
   como más deforestados son los mismos que marcan las demás? Se usa el
   total acumulado de cada fuente por `cod_dane` en toda la ventana
   (2020-2026), el mismo agregado de "Agregado por municipio" más arriba.

En ambos casos se reporta **Pearson** (correlación lineal, sensible a la
magnitud) y **Spearman** (correlación de rangos): dado el fuerte sesgo a la
derecha que ya se encontró en el EDA visual (Hallazgos de esa sección),
Spearman es la medida más confiable acá -- le importa el orden relativo, no
la magnitud, y no lo distorsionan los outliers extremos (Cartagena del
Chairá, San Vicente del Caguán).

In [ ]:
# Cifras nacionales por año, las cuatro fuentes juntas. NaN = ese año
# no existe (o no esta completo) en esa fuente -- no se rellena con 0
# porque "sin dato" y "cero deforestacion" son cosas distintas.
anual_4f = pd.DataFrame({
    "GFW":    gfw.groupby("anio").area_def_ha.sum(),
    "Hansen": hansen.groupby("anio").perdida_ha.sum(),
    "IDEAM":  (ideam.assign(anio=ideam.periodo.str[-4:].astype(int))
                     .groupby("anio").def_ha.sum()),
    "DTD":    dtd.groupby("anio").n_alertas.sum(),
})
anual_4f.round(0)

In [ ]:
# La correlacion solo tiene sentido en los anios donde las cuatro
# fuentes tienen dato -- normalmente excluye 2020 (IDEAM arranca en el
# periodo 2020-2021, que se cuenta como 2021) y el anio en curso (2026,
# parcial: ver fecha_fin en config_local.py).
comun = anual_4f.dropna()
print(f"Anios con dato en las cuatro fuentes: {list(comun.index.astype(int))}")
print()
print("Correlacion de Pearson (nivel):")
print(comun.corr(method="pearson").round(2).to_string())
print()
print("Correlacion de Spearman (rangos -- mas robusta al sesgo):")
print(comun.corr(method="spearman").round(2).to_string())

### Agregado por municipio (todo el periodo)

Mismo total por `cod_dane` que en "Agregado por municipio" más arriba
(`MEDIDA`, definido en esa sección), reutilizado aquí para la correlación
espacial: si dos fuentes están de acuerdo, los municipios con el total más
alto en una deberían ser, en general, los mismos en la otra.

In [ ]:
por_municipio_4f = pd.DataFrame({
    fuente: tabla.groupby("cod_dane")[col].sum()
    for fuente, (tabla, col) in MEDIDA.items()
})
print(f"Municipios: {len(por_municipio_4f):,}")
print()
print("Correlacion de Pearson (nivel):")
print(por_municipio_4f.corr(method="pearson").round(2).to_string())
print()
print("Correlacion de Spearman (rangos):")
print(por_municipio_4f.corr(method="spearman").round(2).to_string())

In [ ]:
from scipy.stats import spearmanr

# Escala log en los dos ejes, porque ya se establecio (seccion de EDA
# visual) que las cuatro distribuciones estan fuertemente sesgadas --
# en escala lineal los outliers aplastarian el resto de los puntos.
# Solo entran municipios con actividad en ambas fuentes del par (>0):
# un municipio en cero en una de las dos no aporta nada en log-log.
#
# ESCALA COMUN: el eje X es la MISMA variable (IDEAM, hectareas) en los
# tres paneles, asi que comparte el mismo rango en los tres -- sin esto,
# el auto-escalado de cada panel usaria un rango distinto (porque el
# conjunto de municipios con match > 0 cambia segun el par), lo que
# distorsionaria visualmente la posicion relativa de cada punto. El eje Y
# comparte rango solo entre GFW y Hansen (las dos en hectareas, igual que
# IDEAM); DTD mide conteo de alertas, otra unidad, y conservar su propio
# eje Y evita una comparacion sin sentido entre hectareas y conteos.
nombres = ideam.drop_duplicates("cod_dane").set_index("cod_dane")["municipio"]
OUTLIERS = ["Cartagena Del Chairá", "San Vicente Del Caguán"]

ideam_pos = por_municipio_4f["IDEAM"]
ideam_pos = ideam_pos[ideam_pos > 0]
xlim_ideam = (ideam_pos.min(), ideam_pos.max())

HECTAREAS_Y = [f for f in HECTAREAS if f != "IDEAM"]  # GFW, Hansen
y_ha = pd.concat([por_municipio_4f[f][por_municipio_4f[f] > 0] for f in HECTAREAS_Y])
ylim_ha = (y_ha.min(), y_ha.max())

fig, ejes = plt.subplots(1, 3, figsize=(14, 4.8))
otras = [f for f in MEDIDA if f != "IDEAM"]
for ax, fuente in zip(ejes, otras):
    x = por_municipio_4f["IDEAM"]
    y = por_municipio_4f[fuente]
    m = (x > 0) & (y > 0)
    rho, _ = spearmanr(x[m], y[m])
    ax.scatter(x[m], y[m], alpha=0.35, s=20, color=FUENTE_COLOR[fuente], edgecolor="none")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*xlim_ideam)
    if fuente in HECTAREAS_Y:
        ax.set_ylim(*ylim_ha)

    # outliers destacados, apilados hacia abajo-izquierda con linea guia
    puntos = []
    for nombre_buscar in OUTLIERS:
        cod = nombres[nombres == nombre_buscar].index
        if len(cod) and cod[0] in x.index and x[cod[0]] > 0 and y[cod[0]] > 0:
            puntos.append((cod[0], x[cod[0]], y[cod[0]]))
    puntos.sort(key=lambda p: -p[2])
    for rank, (cod, xv, yv) in enumerate(puntos):
        ax.scatter([xv], [yv], s=32, color=CRITICAL, zorder=5)
        ax.annotate(nombre_propio(nombres[cod]), xy=(xv, yv),
                    xytext=(-16, -14 - rank * 22), textcoords="offset points",
                    ha="right", va="top", fontsize=8.3, color=CRITICAL, fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color=CRITICAL, linewidth=0.8))

    ax.text(0.04, 0.93, f"Spearman ρ = {rho:.2f}\nn={m.sum():,}", transform=ax.transAxes,
            fontsize=9.5, color=INK_PRIMARY, va="top",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=AXIS, linewidth=0.8))
    ax.set_xlabel("IDEAM def_ha (total municipio)", fontsize=9.5)
    ax.set_ylabel(f"{fuente} (total municipio)", fontsize=9.5)
    ax.set_title(f"IDEAM vs {fuente}", fontsize=11.5, color=INK_PRIMARY, loc="left")
    limpiar_ejes(ax, grid_eje="both")

fig.suptitle("Correlación espacial: ¿coinciden los municipios más deforestados según cada fuente?",
             x=0.03, y=0.99, ha="left", fontsize=13.5, fontweight="bold", color=INK_PRIMARY)
fig.text(0.03, 0.905, "Solo municipios con actividad en ambas fuentes (>0, escala log-log).",
          fontsize=10, color=INK_SECONDARY)
fig.text(0.03, 0.865, "Escala común: eje X compartido (IDEAM) en los tres paneles; eje Y compartido "
          "entre GFW/Hansen (hectáreas) -- DTD aparte (otra unidad: conteo).",
          fontsize=9, color=INK_MUTED, style="italic")
fig.tight_layout(rect=[0, 0, 1, 0.82])

### Hallazgos

- **En el tiempo (nacional, 2021-2025 -- los cinco años con dato en las
  cuatro fuentes; se excluyen 2020, sin IDEAM, y 2026, parcial): GFW,
  Hansen e IDEAM concuerdan razonablemente bien entre sí** (Pearson
  0.69-0.95, Spearman 0.6-0.9) -- las tres bajan desde un pico en 2021
  hasta un mínimo en 2023, con un repunte parcial después. **DTD va en
  sentido contrario**: su correlación con las otras tres es NEGATIVA
  (Pearson entre -0.53 y -0.81; Spearman entre -0.6 y -0.8), porque el
  número de detecciones tempranas *sube* de 2021 a 2025 mientras las
  hectáreas de las otras tres *bajan*. La lectura más probable no es que
  la deforestación esté realmente aumentando según DTD, sino que DTD
  cuenta puntos de alerta (no hectáreas) y es sensible al esfuerzo de
  monitoreo del SMByC, que parece haber crecido en esos años -- por eso
  su tendencia temporal no debería leerse al pie de la letra como una
  señal de deforestación.
- **En el espacio (1,114 municipios, toda la ventana 2020-2026): las
  cuatro fuentes coinciden fuertemente en QUÉ municipios son los más
  deforestados**, incluyendo DTD -- Pearson 0.96-0.99 (inflado por los
  mismos outliers extremos de la sección de EDA visual, Cartagena del
  Chairá y San Vicente del Caguán, que dominan las cuatro fuentes por
  igual) y Spearman, más confiable acá, 0.74-0.81. Esto valida usar
  cualquiera de las cuatro (o una combinación) para el tamizaje de riesgo
  municipal, que es el uso previsto del cruce con las variables
  financieras.
- **Implicación para el modelado:** DTD es confiable para el corte
  espacial (qué municipios) pero no para el corte temporal (qué años) sin
  ajustar por el esfuerzo de monitoreo -- conviene no usarlo tal cual como
  serie de tiempo de deforestación sin controlar por año (p. ej., con
  efectos fijos de año), aunque sí sirve para el ranking de riesgo por
  municipio junto con las otras tres.

---

## 8. Calidad de datos: faltantes, duplicados y outliers

Punto 5 de la lista del tutor, semanas 9-10 del cronograma. Los outliers ya
se documentaron de forma visual/cualitativa en la sección de EDA visual
(Hallazgos, arriba); esta sección completa el reporte formal con tres
verificaciones sistemáticas sobre las cuatro fuentes:

1. **Faltantes** -- distinguiendo los que son estructurales por diseño (por
   ejemplo, las columnas de rezago de GFW no pueden existir en los primeros
   periodos de cada celda) de un hueco real e inesperado en los datos.
2. **Duplicados** -- filas exactamente repetidas, y duplicados de la llave
   `cell_id` + periodo/año (que invalidarían cualquier `.sum()` o `.groupby()`
   ya usado en este cuaderno, incluidas las correlaciones de la sección 7).
3. **Outliers, con una regla estadística explícita** -- percentil 99 entre
   las celda-periodo *con evento* de cada fuente (no sobre el total, que es
   mayormente ceros -- ver % de ceros en la sección de EDA visual), y si esos
   outliers se concentran en los mismos municipios ya identificados
   (Cartagena del Chairá, San Vicente del Caguán).

### 8.1 Faltantes

In [ ]:
FUENTES_DF = {"GFW": gfw, "Hansen": hansen, "IDEAM": ideam, "DTD": dtd}

faltantes = []
for fuente, tabla in FUENTES_DF.items():
    nulos = tabla.isna().sum()
    nulos = nulos[nulos > 0]
    for col, n in nulos.items():
        faltantes.append({
            "fuente": fuente, "columna": col,
            "faltantes": n, "% del total": round(100 * n / len(tabla), 3),
        })

if faltantes:
    print("Columnas con al menos un faltante (las que no aparecen aqui tienen 0):\n")
    print(pd.DataFrame(faltantes).set_index(["fuente", "columna"]).to_string())
else:
    print("Ninguna columna de ninguna fuente tiene faltantes.")

In [ ]:
# GFW tiene columnas de rezago (lag_1_ha, lag_2_ha, lag_3_ha, media_movil_3):
# por construccion, las primeras 1, 2 y 3 celda-mes de cada celda no tienen
# historia suficiente para calcularlas -- eso es un faltante ESTRUCTURAL, no
# un hueco real. Se verifica contando cuantas celdas hay y comparando contra
# el conteo esperado (n_celdas x posicion del rezago).
n_celdas = gfw.cell_id.nunique()
for col, rezago in [("lag_1_ha", 1), ("lag_2_ha", 2), ("lag_3_ha", 3)]:
    esperado = n_celdas * rezago
    real = gfw[col].isna().sum()
    print(f"{col:<14} faltantes reales: {real:>7,}   esperados (n_celdas x {rezago}): {esperado:>7,}   "
          f"coincide: {real == esperado}")

### 8.2 Duplicados

In [ ]:
# cell_id + periodo (o anio, para Hansen) deberia ser una llave unica en
# las cuatro fuentes -- si no lo es, cualquier .sum()/.groupby() ya hecho en
# este cuaderno (incluidas las correlaciones de la seccion 7) estaria
# contando filas de mas.
llave_periodo = {"GFW": "periodo", "Hansen": "anio", "IDEAM": "periodo", "DTD": "periodo"}

for fuente, tabla in FUENTES_DF.items():
    col = llave_periodo[fuente]
    dup_exactas = tabla.duplicated().sum()
    dup_llave = tabla.duplicated(subset=["cell_id", col]).sum()
    print(f"{fuente:<8} filas: {len(tabla):>9,}   duplicadas exactas: {dup_exactas:>4}   "
          f"duplicadas de llave (cell_id + {col}): {dup_llave:>4}")

### 8.3 Outliers -- regla estadística explícita

In [ ]:
# Se define outlier como: celda-periodo CON evento (>0) que supera el
# percentil 99 de esa misma fuente, calculado solo entre las celda-periodo
# con evento -- calcularlo sobre el total no serviria (el total es
# mayormente ceros, ver % de ceros en la seccion de EDA visual, asi que el
# p99 del total seguiria siendo cero o casi cero).
print("Outliers por fuente (p99 calculado solo entre celda-periodo con evento):\n")
for fuente, (tabla, col) in MEDIDA.items():
    con_evento = tabla[tabla[col] > 0]
    p99 = con_evento[col].quantile(0.99)
    outliers = con_evento[con_evento[col] > p99]
    top_municipios = (outliers.groupby("municipio")[col].sum()
                              .sort_values(ascending=False).head(3))
    print(f"{fuente:<8} p99 (con evento) = {p99:>12,.2f} {col:<16} "
          f"outliers: {len(outliers):>5} ({100 * len(outliers) / len(con_evento):.2f}% de las celda-periodo con evento)")
    print("          concentrados en:",
          ", ".join(f"{m} ({v:,.0f})" for m, v in top_municipios.items()))
    print()

### Hallazgos

- **Faltantes: todos estructurales, ninguno es un hueco real.** Los únicos
  NaN son las columnas de rezago de GFW (`lag_1_ha` 1.27%, `lag_2_ha`
  2.53%, `lag_3_ha`/`media_movil_3` 3.80%) -- confirmado exacto contra el
  conteo esperado (n_celdas × posición del rezago, coincide: True en los
  tres) -- y `n_alertas_sinap` de DTD (28.0%), que ya se sabía que el
  SMByC no reporta en todos los trimestres (sección "Detecciones por
  año", arriba: 18 de 25 trimestres con dato). `cod_dane`, `municipio` y
  `departamento` no tienen ningún faltante en ninguna de las cuatro
  fuentes -- el cruce municipal (`asignar_municipio()`) asignó el 100%
  de las celdas.
- **Duplicados: cero, en las cuatro fuentes.** Ni filas exactamente
  repetidas ni duplicados de la llave `cell_id` + periodo/año. Esto
  confirma que las agregaciones (`.sum()`, `.groupby()`) usadas en todo
  el cuaderno -- incluidas las correlaciones de la sección 7 -- no están
  contando ninguna fila de más.
- **Outliers: ~1% de las celda-periodo con evento en cada fuente (por
  definición del p99), pero concentrados de forma consistente en el
  mismo puñado de municipios en las CUATRO fuentes de forma
  independiente.** Cartagena del Chairá y San Vicente del Caguán
  (Caquetá) encabezan en GFW, Hansen e IDEAM; La Macarena (Meta)
  aparece en GFW, Hansen y DTD; Calamar (Guaviare) aparece en IDEAM y
  DTD. Que cuatro mediciones independientes -- con distintos sensores,
  unidades y metodología -- coincidan en los mismos municipios extremos
  es evidencia fuerte de que es un fenómeno real (el "arco de
  deforestación" de la Amazonía colombiana), no un error de datos.
  Confirma la conclusión de la sección de EDA visual: el sesgo no
  requiere limpieza, es una característica genuina de dónde se
  concentra la deforestación.

### 8.4 Decisiones: por qué no se eliminan los outliers ni el sesgo, y qué se imputa

Punto explícito del tutor: declarar y justificar estas decisiones, no solo describirlas.

**Por qué no se eliminan los outliers ni se corrige el sesgo.**

- Los "outliers" de este panel (Cartagena del Chairá, San Vicente del Caguán, La Macarena, Calamar) no son errores de medición: los señalan como los municipios más deforestados **cuatro fuentes independientes** (distinto sensor, distinta metodología -- sección 8.3), y el sesgo a la derecha de la sección 5 es la forma real de un fenómeno que se concentra geográficamente (el "arco de deforestación"), no un artefacto del pipeline.
- Eliminarlos o transformarlos (winsorizing, recorte de percentiles, etc.) en esta etapa iría en contra de dos KPI ya fijados en el documento de entendimiento del negocio (`Entendimiento_del_negocio_OFR_OFICIAL.docx`):
  - **Criterio de éxito de negocio (objetivo 1):** "identificar al menos una cadena productiva o municipio donde coincidan alto financiamiento agropecuario y alto riesgo de deforestación" -- ese "alto riesgo" son justamente los municipios que hoy se ven como outliers. Quitarlos borraría los candidatos que el negocio necesita.
  - **Criterio de éxito de minería (objetivo 1, parte b):** validar el índice propio contra el IDEAM -- la correlación espacial (sección 7) está dominada precisamente por estos municipios extremos; recortarlos cambiaría el número de concordancia sin que eso refleje una mejora real de calidad de datos.
- **Decisión:** no se elimina, recorta ni transforma ningún valor en esta etapa (Fase 2, EDA). Se documentan como hallazgo, y se deja para la Fase 4 (Modelado) decidir un tratamiento estadístico si hace falta -- por ejemplo, modelos robustos a colas pesadas, o una transformación logarítmica solo para visualización, nunca para "corregir" el dato de origen.

**Qué y por qué imputar.**

De los faltantes reales de la sección 8.1, hay dos casos y ninguno se resuelve con un valor numérico fijo:

1. **Rezagos de GFW** (`lag_1_ha`, `lag_2_ha`, `lag_3_ha`, `media_movil_3`): el NaN es estructural -- ocurre solo en los primeros 1 a 3 periodos de cada celda, porque ese "mes anterior" no existe todavía en la ventana de datos (confirmado exacto contra el conteo esperado, sección 8.1). Imputar 0 asumiría "no hubo deforestación" en un periodo que en realidad está fuera de rango, sesgando a la baja cualquier modelo de series de tiempo que use estas columnas. **No se imputa un valor**: se deja el NaN (los modelos de árboles como XGBoost/LightGBM lo manejan nativamente) o, si el modelo elegido no tolera NaN, se agrega una bandera binaria ("sin historial previo") en vez de rellenar con un número.
2. **`n_alertas_sinap` de DTD** (28% NaN): no es un faltante del pipeline propio, es que el SMByC no reportó esa capa en todos los trimestres (18 de 25 trimestres con dato). Imputar 0 asumiría "no hubo alertas" cuando en realidad "no se sabe", sesgando a la baja cualquier análisis que use esta columna. **No se imputa**: se recomienda excluir la columna de cualquier análisis que necesite el periodo completo, o limitar ese análisis puntual a los 18 trimestres con dato, dejándolo documentado.

**Regla general:** en este panel, un NaN siempre significa "no se sabe" o "no aplica", nunca "cero" -- por eso la imputación por defecto (0, media, etc.) queda descartada en los dos casos, y el criterio de minería de datos ("dataset sin datos faltantes críticos") se cumple porque los faltantes están **explicados y acotados**, no porque se hayan tapado con un valor inventado.

---

## 9. Diccionario de variables: los campos más importantes

Documentar las 19 + 9 + 15 + 14 = 57 columnas de las cuatro tablas con el
mismo detalle no es realista ni útil -- es el mismo criterio que pidió el
profesor Jairo para EVA/ICA pecuario (dado el número de fuentes del
proyecto, documentar lo más importante en vez de todo por igual), aplicado
aquí a las cuatro tablas propias de deforestación. Para que la selección no
sea arbitraria, se usa el **mismo criterio explícito ya fijado para
EVA/ICA** (`11_eda_eva_ica`, sección 5), de modo que el
diccionario de variables sea consistente entre las siete fuentes del
proyecto:

**Un campo es prioritario si cumple al menos una de estas tres condiciones:**

1. **Es parte de la llave de cruce** -- sin él no se puede integrar la fuente
   con el resto del panel.
2. **Ya se usa explícitamente** en el panel integrado
   (`06_carga_panel_municipal`).
3. **Es necesario para juzgar la calidad o comparabilidad** de otro campo
   prioritario, o para validar/comparar una fuente contra otra, aunque no sea
   en sí mismo una variable explicativa.

Un campo que no cumple ninguna de las tres no es "poco importante en
general": es poco importante **para las preguntas que este proyecto tiene
planteadas hoy**. Por eso cada tabla también documenta, para los campos no
prioritarios, cuál de las tres condiciones les falta.

**Hallazgo que se desprende de aplicar el criterio -- y que vale la pena
adelantar antes de las cuatro tablas:** de las cuatro fuentes, hoy **solo
GFW e IDEAM entran al panel integrado** (`06_carga_panel_municipal` lee
`PANEL_DEFORESTACION_COLOMBIA` y `PANEL_IDEAM`, nunca `PANEL_HANSEN` ni
`PANEL_DTD` -- verificable en el código de ese notebook). Esto no es un
descuido: coincide exactamente con lo que los propios scripts de descarga ya
documentan como el rol previsto de cada fuente -- Hansen "responde cómo se
compara Colombia con otros países" (sección 2, arriba) y DTD se describe en
su propio script como de "**uso previsto: validación y contexto**", no como
fuente de la medida principal. Por eso, para Hansen y DTD, la condición 2
nunca se cumple hoy -- y eso es evidencia a favor de la clasificación de las
fuentes, no una limitación del criterio.


### 9.1 `PANEL_DEFORESTACION_COLOMBIA` (GFW)

| Campo | Condición que cumple | Por qué |
|---|---|---|
| `cell_id`, `periodo` | 1 (llave) | Sin ellos no hay panel: identifican la celda de 5 km y el mes. |
| `anio`, `cod_dane` | 1 (llave) + 2 (ya se usa) | `anio` es la llave temporal con la que `06_carga_panel_municipal` agrega a nivel municipio-año (`llave = ["cod_dane", "anio"]`); `cod_dane` es la llave de cruce municipal en ese mismo panel. |
| `area_def_ha` | 2 (ya se usa) | Es la medida principal; entra al panel integrado renombrada `alertas_gfw_ha`. |
| `evento` | 3 (calidad/comparabilidad) | Bandera precalculada de "hubo disturbio" (sección 1: "celda-mes con evento: 28.17 %"); documenta de forma explícita el mismo filtro ( > 0) que para Hansen, IDEAM y DTD hay que rehacer a mano en la sección 5 -- sin ella, no quedaría registrado que ese 28.17 % es un número ya verificado, no solo una impresión visual del histograma. |
| `bosque_base_ha` | 3 (comparabilidad) | Es la columna con la que la celda de verificación ("Resumen", arriba) confirma que las cuatro tablas comparten la misma base de bosque -- condición necesaria para poder compararlas entre sí. |
| `mes` | ninguna hoy | Redundante con `periodo` para cualquier cálculo o cruce; no se usa en ningún lado de este cuaderno ni del panel integrado. |
| `lon`, `lat` | 3 (comparabilidad) | Aunque las cuatro tablas tienen su propia copia (idéntica, por compartir grilla), es la copia de **GFW** la que se usa para ubicar las celdas del mapa comparativo de las cuatro fuentes ("Cómo se ve cada fuente en el mapa"). |
| `departamento`, `municipio` | ninguna hoy | Redundantes con `cod_dane` para cualquier cálculo; `municipio` sí se usa para etiquetar impresiones legibles (sección 8.3, "concentrados en..."), pero como texto de reporte, no como variable. |
| `def_acum_ha`, `bosque_remanente_ha`, `tasa_def` | ninguna hoy | Quedan calculadas en el panel pero no se leen en ningún punto de este cuaderno ni del panel integrado -- candidatas a revisar si se necesita una medida de deforestación *relativa* al bosque restante, en vez de absoluta. |
| `lag_1_ha`, `lag_2_ha`, `lag_3_ha`, `media_movil_3` | ninguna hoy (candidatas) | Features de series de tiempo; su único uso en este cuaderno es confirmar que su NaN es estructural (sección 8.1). Candidatas directas para un modelo de series de tiempo en Fase 3 -- mismo tipo de caso que `Produccion`/`Rendimiento` en el diccionario de EVA. |


### 9.2 `PANEL_HANSEN`

| Campo | Condición que cumple | Por qué |
|---|---|---|
| `cell_id`, `anio`, `cod_dane` | 1 (llave) | Mismo rol que en GFW; permiten agrupar por celda, año y municipio dentro de este cuaderno (aunque Hansen no entra hoy al panel integrado -- ver nota de arriba). |
| `perdida_ha` | 3 (comparabilidad) | No se usa en `06_carga_panel_municipal` (condición 2 no se cumple -- ver nota de arriba), pero es indispensable **dentro de este cuaderno**: es uno de los cuatro términos de la correlación espacial (sección 7) y uno de los cuatro conjuntos de outliers que, al coincidir en los mismos municipios (sección 8.3), sostiene la decisión de no limpiarlos (sección 8.4). Sin esta columna no se podría validar a GFW/IDEAM/DTD de forma independiente. |
| `bosque_base_ha` | 1 + 3 (origen y comparabilidad) | Hansen GFC es, de hecho, la **fuente** del `bosque_base_ha` compartido por las cuatro tablas (`calcular_bosque.py` reproyecta esta misma capa sobre la grilla nacional); es la razón de que la verificación de "misma base común" (sección "Resumen") sea posible. |
| `lon`, `lat`, `departamento`, `municipio` | ninguna hoy | Redundantes: el mapa usa la copia de GFW (idéntica); `municipio` solo aparece en los mismos textos de reporte que en las otras tres fuentes. |


### 9.3 `PANEL_IDEAM`

| Campo | Condición que cumple | Por qué |
|---|---|---|
| `cell_id`, `periodo`, `anio`, `cod_dane` | 1 (llave) + 2 (ya se usa) | Las cuatro entran al panel integrado: `cell_id` para contar `n_celdas` por municipio, `periodo`/`anio` como parte de la llave `["cod_dane", "anio"]`, `cod_dane` como llave de cruce. |
| `def_ha` | 2 (ya se usa) | La medida oficial; entra al panel integrado renombrada `def_ideam_ha`. |
| `bosque_ideam_ha` | 2 (ya se usa) | Denominador correcto de la tasa de deforestación (sección 3: "el denominador correcto... es `bosque_ideam_ha`"); se usa directamente y además para `bosque_ideam_inicial_ha` en el panel integrado. |
| `sin_info_ha` | 2 (ya se usa) | Entra al panel integrado renombrada `sin_info_ideam_ha` -- es la columna de control de calidad (nubosidad) de la propia leyenda del IDEAM. |
| `bosque_estable_ha`, `regeneracion_ha` | 3 (comparabilidad) | Se usan en la celda de la sección 3 que reconstruye la cifra oficial publicada por periodo y la contrasta contra el dato ("recuperado_%", 99.65-99.84 %) -- es decir, son necesarias para *validar* que `def_ha` agrega correctamente, no para explicar nada por sí mismas. |
| `departamento` | 3 (comparabilidad) | Se usa para el boxplot de dispersión departamental (sección 5): necesaria para juzgar si la concentración municipal de los outliers (sección 8.3) es un patrón homogéneo a nivel departamental o no (resulta que no -- hay mucha dispersión intra-departamental). |
| `no_bosque_ha`, `bosque_ideam_fin_ha` | ninguna hoy | Completan la leyenda de las cinco clases del SMByC, pero ninguna de las dos se usa en ningún cálculo de este cuaderno. `bosque_ideam_fin_ha` sería el insumo natural si se quisiera calcular una tasa de cambio propia de cada transición, en vez de usar `bosque_ideam_ha` del periodo inicial. |
| `lon`, `lat`, `municipio` | ninguna hoy | Mismo patrón que en las otras tres fuentes: redundantes o solo para texto de reporte. |
| `bosque_base_ha` | 3 (comparabilidad) | Igual que en GFW y Hansen: verificación de base común. |


### 9.4 `PANEL_DTD`

| Campo | Condición que cumple | Por qué |
|---|---|---|
| `cell_id`, `cod_dane` | 1 (llave) | Identifican la celda y permiten el cruce municipal (aunque DTD no entra hoy al panel integrado -- ver nota de arriba). |
| `periodo`, `trimestre_txt`, `anio`, `trimestre` | 1 (llave) | Entre las cuatro cubren el mismo grano temporal en tres formatos (fecha, texto, año, número de trimestre); `trimestre_txt` es la que se usa para la comprobación de panel balanceado y el rango de cobertura (sección 4: "2020-T1 a 2026-T1"). |
| `n_alertas` | 3 (comparabilidad) | No se usa en el panel integrado (condición 2 no se cumple), pero es uno de los cuatro términos de la correlación espacial (sección 7) y de la concentración municipal de outliers (sección 8.3) -- mismo rol de validación cruzada que `perdida_ha` de Hansen. |
| `n_alertas_sinap`, `sinap_disponible` | 3 (comparabilidad) | `sinap_disponible` es la bandera que distingue "no hubo alertas en área protegida" de "el SMByC no reportó esa capa ese trimestre" -- sin ella no se podría tomar la decisión de no imputar el 28 % de `n_alertas_sinap` que aparece como faltante (secciones 8.1 y 8.4). Es decir, es necesaria para juzgar correctamente la calidad de otro campo (`n_alertas_sinap`), aunque ninguna de las dos sea en sí misma una variable explicativa. |
| `bosque_base_ha` | 3 (comparabilidad) | Igual que en las otras tres fuentes. |
| `lon`, `lat`, `departamento`, `municipio` | ninguna hoy | Mismo patrón: redundantes o solo para texto de reporte. |


**Síntesis del diccionario.**

El patrón que deja este ejercicio es consistente con la síntesis de la
sección 10 (más abajo): las columnas que cumplen la condición 2 --
`area_def_ha` de GFW y `def_ha` / `bosque_ideam_ha` / `sin_info_ha` de IDEAM
-- son exactamente las que hoy alimentan el cruce con financiamiento
agropecuario que persigue el objetivo de minería 2 del documento de
entendimiento del negocio. Hansen y DTD no tienen ninguna columna en
condición 2 todavía, pero sus medidas principales (`perdida_ha`,
`n_alertas`) sí cumplen la condición 3 de forma consistente: son el material
con que este mismo cuaderno valida que GFW e IDEAM señalan los mismos
municipios de alto riesgo (sección 7 y 8.3) -- que es, de hecho, el
fundamento empírico de la correlación que contribuye al criterio de éxito de
minería de validar el índice propio contra el IDEAM (sección 10.1, más
abajo). En otras palabras: que una fuente no entre hoy al panel integrado no
significa que esté fuera de alcance -- Hansen y DTD cumplen exactamente el
rol de "validación y contexto" que sus propios scripts de descarga ya les
asignan, y ese rol es el que documenta la condición 3.


---

## 10. Síntesis: insights del EDA frente a los KPI del proyecto

Con el EDA de las cuatro tablas propias completo (secciones 5 a 8), esta sección conecta esos hallazgos con los KPI ya definidos en el documento de entendimiento del negocio (`Entendimiento_del_negocio_OFR_OFICIAL.docx`) y deja un catálogo de qué alimenta cada gráfico -- los dos puntos que pedía el tutor. **Alcance:** solo las cuatro tablas de deforestación; lo que dependa de cruzar con las variables financieras del profesor (FINAGRO, SFC414) queda marcado como pendiente de Fase 3.

In [ ]:
import matplotlib.patches as mpatches

# Cuatro numeros que resumen la seccion 10.1 (mas abajo) en una sola figura,
# como pidio el tutor ("graficos que llamen la atencion con kpis").
TILES = [
    ("99.3%", "de los municipios de Colombia\ncubiertos por el panel (1,114 de 1,122)", GOOD),
    ("0%", "de faltantes en la llave municipal\n(cod_dane, municipio, departamento)", GOOD),
    ("4 de 4", "fuentes coinciden en los municipios\nde mayor riesgo (sección 7, Hallazgos)", BLUE),
    ("0.74–0.81", "correlación espacial (Spearman)\nentre las cuatro fuentes", BLUE),
]

fig, ejes = plt.subplots(1, 4, figsize=(13, 3.1))
for ax, (valor, etiqueta, color) in zip(ejes, TILES):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(mpatches.Rectangle((0.0, 0.97), 0.22, 0.03, transform=ax.transAxes,
                                     facecolor=color, edgecolor="none", clip_on=False))
    ax.text(0.0, 0.72, valor, transform=ax.transAxes, fontsize=30, fontweight="bold",
            color=INK_PRIMARY, ha="left", va="top")
    ax.text(0.0, 0.38, etiqueta, transform=ax.transAxes, fontsize=10.5,
            color=INK_SECONDARY, ha="left", va="top", linespacing=1.4)

fig.suptitle("EDA de deforestación — KPI del proyecto en cuatro números", x=0.015, y=0.98,
             ha="left", fontsize=14.5, fontweight="bold", color=INK_PRIMARY)
fig.text(0.015, 0.88,
          "Frente a los criterios de éxito de negocio y de minería de datos del documento de entendimiento del negocio (sección 10.1).",
          fontsize=10, color=INK_SECONDARY)
fig.tight_layout(rect=[0.01, 0.02, 0.99, 0.78])

### 10.1 Insight de los EDA asociados a los KPI de negocio y de minería

**KPI de negocio:**

- *Objetivo 1 -- "identificar al menos una cadena productiva o municipio donde coincidan alto financiamiento agropecuario y alto riesgo de deforestación":* el EDA ya resuelve la mitad de "alto riesgo de deforestación" -- Cartagena del Chairá y San Vicente del Caguán (Caquetá), La Macarena (Meta) y Calamar (Guaviare) aparecen consistentemente como los municipios de mayor deforestación en las cuatro fuentes (sección 8.3). Falta la otra mitad ("alto financiamiento agropecuario"), que depende de cruzar contra FINAGRO -- pendiente de Fase 3.
- *Objetivo 2 -- "caso de éxito reutilizable, dejando conocimiento y metodología reutilizables":* el EDA mismo es evidencia de esto -- código versionado, reproducible, con Hallazgos documentados en cada sección y trazabilidad de cada transformación (`GUIA_CODIGO.md`), en línea con el "rigor metodológico" que el documento cita como uno de los cuatro principios del Observatorio.

**KPI de minería:**

- *Objetivo 1, parte a -- "dataset final sin datos faltantes críticos, con cobertura de al menos el 60% de los municipios":* **cumplido.** `cod_dane`, `municipio` y `departamento` no tienen ningún faltante en las cuatro fuentes (asignación municipal al 100%, sección 8.1); los únicos NaN reales (rezagos de GFW, trimestres sin reporte DTD) están explicados y acotados (ver 8.4), no comprometen la variable objetivo. Cobertura: 1,114 de los 1,122 municipios de Colombia (~99.3%), muy por encima del umbral del 60%.
- *Objetivo 1, parte b -- "validar el índice de deforestación del equipo contra el IDEAM/SMByC, alcanzando al menos 80% de concordancia municipio a municipio, y contra la definición de la EUDR, documentando divergencias":* **parcialmente medido, con una ambigüedad a resolver con el tutor.** La correlación espacial de Spearman entre IDEAM y las otras tres fuentes (sección 7) da 0.74-0.81 -- en el borde del umbral, pero una *correlación* no es necesariamente lo mismo que una *concordancia* (por ejemplo, % de municipios donde ambas fuentes coinciden en la misma categoría de riesgo alto/bajo); falta acordar cuál de las dos es la métrica que se va a reportar como KPI. La comparación contra la definición de bosque/deforestación de la EUDR (documentar los casos que divergen, cobertura 10-29% o usos no agrícolas) todavía no se ha hecho en el EDA -- solo se investigó a nivel conceptual.
- *Objetivo 2 -- "ranking de las tres variables (top 3) con mayor asociación significativa (p < 0,05) con la deforestación":* **fuera de alcance de este EDA**, porque exige variables financieras/de producción (FINAGRO, SFC414), que están fuera del alcance actual del equipo. Lo que sí se hizo en su lugar (sección 7) fue correlacionar las cuatro fuentes de deforestación *entre sí* -- eso valida qué tan confiable es la variable objetivo, pero no responde a este objetivo de minería, que solo se puede cerrar en la Fase 3.

### 10.2 Cómo se alimenta cada gráfico y qué insight comunica

| Gráfico | Dónde | De dónde sale / transformación | Insight que comunica |
|---|---|---|---|
| Mapa de calor, una capa por fuente | "Cómo se ve cada fuente en el mapa" | Cada fuente se agrupa por `cell_id` sumando toda la ventana 2020-2026; se toman las 4,000 celdas de mayor actividad por fuente y el peso se normaliza al máximo de esa misma fuente (0-1), para que unidades distintas (hectáreas vs. conteos) no se comparen directamente. | Si los focos de calor de las cuatro capas coinciden geográficamente, ya es una primera señal visual de que las fuentes están de acuerdo en el patrón espacial -- lo que la sección 7 después cuantifica con números. |
| Histograma de la medida principal, celda × periodo (log-y) | 5. Distribución de cada fuente | Para cada fuente se filtran solo los registros con valor > 0 de su medida principal y se grafican en escala log en el eje Y. | Muestra la forma real de la distribución cuando sí hay evento: cola larga a la derecha (pocas celdas con mucha área/alertas). El sesgo es el hallazgo en sí, no un error de datos (ver 8.4). |
| Histograma de la medida principal, agregado por municipio (log-y) | 5. Distribución de cada fuente | Suma de la medida principal por `cod_dane`, luego histograma en escala log. | El mismo sesgo aparece a nivel municipal: la deforestación se concentra en pocos municipios, anticipando los outliers de la sección 8.3. |
| Boxplot de dispersión departamental (IDEAM, top 8) | 5. Distribución de cada fuente | Suma `def_ha` por municipio, agrupada por departamento; se toman los 8 departamentos con mayor total acumulado. | Incluso dentro de los departamentos más deforestados hay mucha dispersión entre sus municipios -- no es un fenómeno homogéneo a nivel departamental. |
| Dispersión log-log, IDEAM vs. cada una de las otras tres fuentes | 7. Correlación entre fuentes (espacial) | Total acumulado 2020-2026 por `cod_dane` de cada fuente, solo municipios con actividad (>0) en ambas fuentes del par; ejes en escala log; Spearman rho de cada par en el título. | Visualiza qué tan alineadas están las fuentes en señalar los mismos municipios de alto riesgo -- los outliers de la sección 8.3 (Cartagena del Chairá, San Vicente del Caguán) se ven como los puntos en la esquina superior derecha de las tres gráficas. |

Dos salidas relacionadas se reportan como **tabla**, no como gráfico, y vale la pena dejarlo explícito porque también responde al punto del tutor: la correlación temporal año a año (sección 7, matrices de Pearson y Spearman) y las cifras nacionales por año lado a lado (sección "Las tres fuentes en hectáreas"). Con solo 5 años de datos comunes a las cuatro fuentes, una matriz de números es más precisa de leer que un gráfico de líneas, que con tan pocos puntos no aporta más información.